# Module 2 Wedding Planner Demo

This notebook displays the saved PowerShell wedding run, state snapshots, Kiwi discovery, and nested callback trace. It does not start MCP, model, Tavily, or database calls.

## Roles and safety boundaries

The coordinator owns the shared state and delegates to travel, venue, and DJ specialists. Travel uses only Kiwi `search-flight`; venue uses Tavily with two searches; DJ uses read-only SQL against Chinook.db. The project never books, reserves, purchases, follows booking links, or invokes Kiwi `feedback-to-devs`.

In [1]:
import json
import sys
from pathlib import Path
from IPython.display import Code, Markdown, display

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / '.venv' / 'Scripts' / 'python.exe').exists():
    project_root = project_root.parent
assert Path(sys.executable).resolve() == (project_root / '.venv' / 'Scripts' / 'python.exe').resolve()
assert sys.version_info[:3] == (3, 13, 1)
project_dir = project_root / 'my-work' / 'module2_wedding_project'
output_path = project_dir / 'wedding_demo_output.txt'
state_path = project_dir / 'wedding_state_snapshots.json'
discovery_path = project_dir / 'kiwi_tool_discovery.json'
trace_path = project_dir / 'traces' / 'wedding_demo_trace_timeline.txt'
for path in (output_path, state_path, discovery_path, trace_path):
    if not path.exists():
        raise FileNotFoundError(f'Missing saved artifact: {path}')
display(Markdown('## Saved PowerShell output'))
display(Code(output_path.read_text(encoding='utf-8'), language='text'))
display(Markdown('## State snapshots'))
display(Code(state_path.read_text(encoding='utf-8'), language='json'))
display(Markdown('## Kiwi discovery summary'))
discovery = json.loads(discovery_path.read_text(encoding='utf-8'))
safe_discovery = {key: discovery.get(key) for key in ('status', 'transport', 'endpoint', 'allowed_travel_tools', 'rejected_tools')}
display(Code(json.dumps(safe_discovery, indent=2), language='json'))
display(Markdown('## Nested callback trace timeline'))
display(Code(trace_path.read_text(encoding='utf-8'), language='text'))

## Saved PowerShell output

Kiwi discovery:
{
  "status": "ok",
  "transport": "streamable_http",
  "endpoint": "https://mcp.kiwi.com",
  "allowed_travel_tools": [
    "search-flight"
  ],
  "rejected_tools": [
    "feedback-to-devs"
  ],
  "tools": [
    {
      "name": "search-flight",
      "description": "# Search for flights\n\nSearches Kiwi.com for available flights between two locations for the given dates and passengers. City or airport names are resolved automatically, so call this whenever the user wants to search for flights — whether they gave IATA codes or just place names.\n\n## Result shape\n\nReturns `{ query, currency, passengers, resultsCount, itineraries, searchTimeMs }`. Each item in `itineraries` has:\n- `price` (number) and `priceFormatted` (e.g. \"123 EUR\")\n- `totalDurationSeconds`\n- `bookingUrl` — the link to book the flight\n- `imageId` — destination city id for a hero photo (https://images.kiwi.com/photos/600x600/{imageId}.jpg)\n- `baggage` — total included baggage across all travelers: `{ personalItem, cabinBag, checkedBag }` (counts)\n- `outbound` (and `inbound` for return flights), each a leg with: `route` (list of airport codes including layovers, e.g. [\"PRG\",\"MAD\",\"BCN\"]), `departureTime` / `arrivalTime` (local ISO timestamps), `durationSeconds`, `stops`, `cabinClass`, and a `segments` list. Each segment has `from`/`to` (IATA), `fromCity`/`toCity`, `fromName`/`toName` (full airport names), `fromCountry`/`toCountry`, `carrier` (code) and `carrierName` (airline name), `flightNumber`, and `cabinClass`.\n\n## How to display\n\nRender a markdown table. Group results into cheapest (lowest `price`), shortest (lowest `totalDurationSeconds`), and other notable options. For each itinerary show, in order:\n  - Route: join `outbound.route` with \" → \" (e.g. \"PRG → MAD → BCN\"); for return flights show `inbound.route` on a second line.\n  - Times & duration: `outbound.departureTime` → `outbound.arrivalTime` (local), formatted readably (e.g. \"12 Jun 06:05 → 09:30 (3h 25m)\") using `outbound.durationSeconds`; add the `inbound` equivalents for return flights.\n  - Cabin: `outbound.cabinClass` (e.g. \"Economy\").\n  - Price: `priceFormatted`.\n  - Booking link: `bookingUrl`.\n\nFinally, summarise the best price and the shortest option, give a recommendation, and end wishing the user a nice trip with a short fun fact about the destination!",
      "args": {
        "flyFrom": {
          "description": "Origin: IATA code, city or airport name (e.g. \"PRG\", \"Prague\", \"London Heathrow\")",
          "maxLength": 100,
          "minLength": 1,
          "title": "Flyfrom",
          "type": "string"
        },
        "flyTo": {
          "description": "Destination: IATA code, city or airport name",
          "maxLength": 100,
          "minLength": 1,
          "title": "Flyto",
          "type": "string"
        },
        "departureDate": {
          "description": "Departure date in dd/mm/yyyy — the day to fly, or the earliest date of a range (see departureDateTo)",
          "maxLength": 10,
          "title": "Departuredate",
          "type": "string"
        },
        "departureDateFlexDays": {
          "default": 0,
          "description": "Search +/- this many days around departureDate (symmetric). Leave 0 when using departureDateTo",
          "maximum": 10,
          "minimum": 0,
          "title": "Departuredateflexdays",
          "type": "integer"
        },
        "departureDateTo": {
          "anyOf": [
            {
              "maxLength": 10,
              "type": "string"
            },
            {
              "type": "null"
            }
          ],
          "default": null,
          "description": "Latest departure date in dd/mm/yyyy for an explicit range [departureDate .. departureDateTo]. Use instead of departureDateFlexDays for a one-directional or asymmetric range; must be >= departureDate",
          "title": "Departuredateto"
        },
        "returnDate": {
          "anyOf": [
   

## State snapshots

{
  "before_initial": {
    "origin": null,
    "destination": null,
    "event_date": null,
    "outbound_date": null,
    "return_date": null,
    "guest_count": null,
    "flight_adults": null,
    "genre": null,
    "brief_status": null,
    "travel_status": null,
    "travel_searches_used": null,
    "venue_status": null,
    "venue_searches_used": null,
    "dj_status": null
  },
  "after_brief_saved_before_delegation": {
    "origin": "Algiers",
    "destination": "Paris",
    "event_date": "15/05/2027",
    "outbound_date": "13/05/2027",
    "return_date": "17/05/2027",
    "guest_count": 30,
    "flight_adults": 2,
    "genre": "Jazz",
    "brief_status": "complete",
    "travel_status": "not_started",
    "travel_searches_used": 0,
    "venue_status": "not_started",
    "venue_searches_used": 0,
    "dj_status": "not_started"
  },
  "after_initial": {
    "origin": "Algiers",
    "destination": "Paris",
    "event_date": "2027-05-15",
    "outbound_date": "2027-05-13",
    "return_date": "2027-05-17",
    "guest_count": 30,
    "flight_adults": 2,
    "genre": "Jazz",
    "brief_status": "complete",
    "travel_status": "completed_search_only",
    "travel_searches_used": 1,
    "venue_status": "completed_candidates",
    "venue_searches_used": 2,
    "dj_status": "completed_database_query"
  },
  "after_follow_up": {
    "origin": "Algiers",
    "destination": "Paris",
    "event_date": "2027-05-15",
    "outbound_date": "2027-05-13",
    "return_date": "2027-05-17",
    "guest_count": 30,
    "flight_adults": 2,
    "genre": "Rock",
    "brief_status": "complete",
    "travel_status": "completed_search_only",
    "travel_searches_used": 1,
    "venue_status": "completed_candidates",
    "venue_searches_used": 2,
    "dj_status": "completed_database_query"
  },
  "travel_searches_repeated_on_follow_up": false,
  "venue_searches_repeated_on_follow_up": false
}

## Kiwi discovery summary

{
  "status": "ok",
  "transport": "streamable_http",
  "endpoint": "https://mcp.kiwi.com",
  "allowed_travel_tools": [
    "search-flight"
  ],
  "rejected_tools": [
    "feedback-to-devs"
  ]
}

## Nested callback trace timeline

001 chain_start chain run=01a101c0-0f55-7910-a64c-e208aa8adca0 parent=root
002   chain_start chain run=01a101c0-0f5a-7453-ac0f-2e8bee34a34d parent=01a101c0-0f55-7910-a64c-e208aa8adca0
003     chat_model_start ChatGoogleGenerativeAI run=01a101c0-0f61-73d0-a754-17a00cff1ff2 parent=01a101c0-0f5a-7453-ac0f-2e8bee34a34d
004     llm_end llm run=01a101c0-0f61-73d0-a754-17a00cff1ff2 parent=01a101c0-0f5a-7453-ac0f-2e8bee34a34d
005   chain_end chain run=01a101c0-0f5a-7453-ac0f-2e8bee34a34d parent=01a101c0-0f55-7910-a64c-e208aa8adca0
006   chain_start chain run=01a101c0-1394-76e3-9fb0-7cd1b8b2bd32 parent=01a101c0-0f55-7910-a64c-e208aa8adca0
007     tool_start update_brief run=01a101c0-1395-7dd0-bdec-62e94f911c49 parent=01a101c0-1394-76e3-9fb0-7cd1b8b2bd32
008     tool_end tool run=01a101c0-1395-7dd0-bdec-62e94f911c49 parent=01a101c0-1394-76e3-9fb0-7cd1b8b2bd32
009   chain_end chain run=01a101c0-1394-76e3-9fb0-7cd1b8b2bd32 parent=01a101c0-0f55-7910-a64c-e208aa8adca0
010   chain_start chain run=01a101c0-139b-7d60-86fe-2bd43c06083d parent=01a101c0-0f55-7910-a64c-e208aa8adca0
011     chat_model_start ChatGoogleGenerativeAI run=01a101c0-139c-7743-8ee1-b61ab8bd5aca parent=01a101c0-139b-7d60-86fe-2bd43c06083d
012     llm_end llm run=01a101c0-139c-7743-8ee1-b61ab8bd5aca parent=01a101c0-139b-7d60-86fe-2bd43c06083d
013   chain_end chain run=01a101c0-139b-7d60-86fe-2bd43c06083d parent=01a101c0-0f55-7910-a64c-e208aa8adca0
014   chain_start chain run=01a101c0-1862-74f2-9522-2388a9f6b8e0 parent=01a101c0-0f55-7910-a64c-e208aa8adca0
015     tool_start search_flights run=01a101c0-1862-74f2-9522-2390a90d4d3b parent=01a101c0-1862-74f2-9522-2388a9f6b8e0
016       chain_start chain run=01a101c0-1869-79c3-8ff9-94d6f91442bb parent=01a101c0-1862-74f2-9522-2390a90d4d3b
017         chain_start chain run=01a101c0-186b-7343-898e-22bd479ff6af parent=01a101c0-1869-79c3-8ff9-94d6f91442bb
018           chat_model_start ChatGoogleGenerativeAI run=01a101c0-186d-7680-ba96-32dcac08cfeb parent=01a101c0-186b-7343-898e-22bd479ff6af
019           llm_end llm run=01a101c0-186d-7680-ba96-32dcac08cfeb parent=01a101c0-186b-7343-898e-22bd479ff6af
020         chain_end chain run=01a101c0-186b-7343-898e-22bd479ff6af parent=01a101c0-1869-79c3-8ff9-94d6f91442bb
021         chain_start chain run=01a101c0-1b28-73e3-9dec-5a35ee971550 parent=01a101c0-1869-79c3-8ff9-94d6f91442bb
022           tool_start search_flights_from_brief run=01a101c0-1b29-7120-8338-899e19022180 parent=01a101c0-1b28-73e3-9dec-5a35ee971550
023             tool_start search-flight run=01a101c0-1b2a-7d12-b2ec-17a058e1b354 parent=01a101c0-1b29-7120-8338-899e19022180
024             tool_end tool run=01a101c0-1b2a-7d12-b2ec-17a058e1b354 parent=01a101c0-1b29-7120-8338-899e19022180
025           tool_end tool run=01a101c0-1b29-7120-8338-899e19022180 parent=01a101c0-1b28-73e3-9dec-5a35ee971550
026         chain_end chain run=01a101c0-1b28-73e3-9dec-5a35ee971550 parent=01a101c0-1869-79c3-8ff9-94d6f91442bb
027         chain_start chain run=01a101c0-29da-7df0-92bd-411e7c763e54 parent=01a101c0-1869-79c3-8ff9-94d6f91442bb
028           chat_model_start ChatGoogleGenerativeAI run=01a101c0-29e0-7fc2-b3d2-d9cdd9db7edc parent=01a101c0-29da-7df0-92bd-411e7c763e54
029           llm_end llm run=01a101c0-29e0-7fc2-b3d2-d9cdd9db7edc parent=01a101c0-29da-7df0-92bd-411e7c763e54
030         chain_end chain run=01a101c0-29da-7df0-92bd-411e7c763e54 parent=01a101c0-1869-79c3-8ff9-94d6f91442bb
031       chain_end chain run=01a101c0-1869-79c3-8ff9-94d6f91442bb parent=01a101c0-1862-74f2-9522-2390a90d4d3b
032     tool_end tool run=01a101c0-1862-74f2-9522-2390a90d4d3b parent=01a101c0-1862-74f2-9522-2388a9f6b8e0
033   chain_end chain run=01a101c0-1862-74f2-9522-2388a9f6b8e0 parent=01a101c0-0f55-7910-a64c-e208aa8adca0
034   chain_start chain run=01a101c0-30e1-7eb0-b224-10236992686c parent=01a101c0-0f55-7910-a64c-e208aa8adca0
035     chat_model_start ChatGoogleGenerativeAI run=01a101c0-30e3-7900-9f18-ae6947a68de4 parent=01a101c0-30e1-7eb0-b224-10236992

## Observations

- Actual failures: Windows Unicode output, missing coordinator checkpointer, and nested-specialist state inheritance.
- Corrective rules: UTF-8 PowerShell execution, one reused `InMemorySaver`, and explicit brief-field passing into specialists.
- Venue limitation: candidates and snippets are not confirmations of capacity, price, or availability.
- Kiwi limitation: `search-flight` results may include booking URLs, but this project never follows or uses them.